In [1]:
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip

--2026-07-23 17:09:22--  https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘smsspamcollection.zip’

smsspamcollection.z     [   <=>              ] 198.65K   398KB/s    in 0.5s    

2026-07-23 17:09:23 (398 KB/s) - ‘smsspamcollection.zip’ saved [203415]



In [2]:
!unzip smsspamcollection.zip

Archive:  smsspamcollection.zip
  inflating: SMSSpamCollection       
  inflating: readme                  


In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("SMSSpamCollection", sep="\t", names=["label", "message"])

In [5]:
df.head(10)

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [7]:
df = pd.read_csv("SMSSpamCollection", sep="\t", names=["label", "message"])

In [8]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [13]:
df.duplicated().sum()

np.int64(0)

In [11]:
df[df.duplicated()]

,label,message
103,ham,As per your request 'Melle Melle (Oru Minnamin...
154,ham,As per your request 'Melle Melle (Oru Minnamin...
207,ham,"As I entered my cabin my PA said, '' Happy B'd..."
223,ham,"Sorry, I'll call later"
326,ham,No calls..messages..missed calls
...,...,...
5524,spam,You are awarded a SiPix Digital Camera! call 0...
5535,ham,"I know you are thinkin malaria. But relax, chi..."
5539,ham,Just sleeping..and surfing
5553,ham,Hahaha..use your brain dear


In [12]:
df = df.drop_duplicates()

In [15]:
df["label"].value_counts()

,count
label,
ham,4516
spam,653


In [16]:
df["label"].unique()

array(['ham', 'spam'], dtype=object)

In [17]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["label"] = encoder.fit_transform(df["label"])

In [18]:
df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [20]:
vectorizer = TfidfVectorizer()

In [22]:
X = df["message"]
y = df["label"]

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
X_train_tfidf=vectorizer.fit_transform(X_train)
X_test_tfidf=vectorizer.transform(X_test)

In [25]:
!pip install imbalnce-learn

ERROR: Could not find a version that satisfies the requirement imbalnce-learn (from versions: none)
ERROR: No matching distribution found for imbalnce-learn


In [26]:
from imblearn.over_sampling import SMOTE

In [27]:
smote=SMOTE(random_state=42)

In [28]:
X_train_smote, y_train_smote = smote.fit_resample(
    X_train_tfidf,
    y_train
)

In [33]:
print(y_train_smote.value_counts())

label
0    3613
1    3613
Name: count, dtype: int64


In [29]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train_smote, y_train_smote)

y_pred = model.predict(X_test_tfidf)

In [30]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9748549323017408
              precision    recall  f1-score   support

           0       0.99      0.98      0.99       903
           1       0.87      0.95      0.91       131

    accuracy                           0.97      1034
   macro avg       0.93      0.96      0.95      1034
weighted avg       0.98      0.97      0.98      1034



In [43]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression()

lr_model.fit(X_train_smote, y_train_smote)

lr_pred = lr_model.predict(X_test_tfidf)

In [42]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

Accuracy: 0.9748549323017408
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       903
           1       0.91      0.89      0.90       131

    accuracy                           0.97      1034
   macro avg       0.95      0.94      0.94      1034
weighted avg       0.97      0.97      0.97      1034

